# 01 - Data Exploration

Load the international results dataset using `src/data.py` and take a first look at it.

In [ ]:
import sys

sys.path.append("..")

from src.data import load_results, run_sanity_checks

In [ ]:
df = load_results()
run_sanity_checks(df)

In [ ]:
df.head()

In [ ]:
df.describe(include="all")

## Elo ratings

Compute leakage-free, rolling Elo ratings for every team using `src/elo.py`, then
look at the top 20 teams by current (most recent) rating.

In [ ]:
from src.elo import compute_elo_ratings, top_teams

df_elo, current_ratings = compute_elo_ratings(df)
df_elo[["date", "home_team", "away_team", "home_elo_pre", "away_elo_pre"]].tail()

In [ ]:
top_teams(current_ratings, n=20)

## Validate the K-factor mapping

Data-quality check (no changes to `elo.py` here): look at how `get_k_factor()`
classifies the `tournament` values that actually occur in results.csv, and flag
any high-frequency tournament that lands in the generic `K_OTHER` bucket but
arguably deserves a higher tier (i.e. its name doesn't exactly match anything
in `MAJOR_TOURNAMENTS`, even though it is a competitive, high-stakes
tournament).

In [ ]:
from src.elo import K_OTHER, get_k_factor

tournament_counts = df["tournament"].value_counts().rename_axis("tournament").reset_index(name="count")
tournament_counts["k_factor"] = tournament_counts["tournament"].apply(get_k_factor)
tournament_counts

In [ ]:
TOP_N = 30

top_tournaments = tournament_counts.head(TOP_N)
print(f"K-factor assigned to the {TOP_N} most common tournaments:")
top_tournaments

In [ ]:
# Among the most common tournaments, which ones land in the generic K_OTHER
# bucket (i.e. didn't match FIFA World Cup, MAJOR_TOURNAMENTS, "qualification",
# or Friendly)?
other_bucket = top_tournaments[top_tournaments["k_factor"] == K_OTHER]
print(f"High-frequency tournaments (top {TOP_N}) mapped to the generic K_OTHER bucket:")
other_bucket

In [ ]:
# Flag entries in that bucket whose name suggests a competitive, high-stakes
# tournament (e.g. a confederation "Nations League") that just doesn't happen
# to match the exact strings in MAJOR_TOURNAMENTS.
flag_keywords = ["Nations League"]
flagged = other_bucket[
    other_bucket["tournament"].str.contains("|".join(flag_keywords), case=False)
]

print("Flagged for review -- arguably major, but currently mapped to K_OTHER:")
print(flagged[["tournament", "count", "k_factor"]].to_string(index=False))
